In [1]:
# DQN (Deep Q-Network) — Chapter 6
# 核心思想：价值迭代 + 神经网络 + 经验回放 + 目标网络
from collections import namedtuple, deque
import numpy as np
import torch
import torch.nn as nn
import random

# 超参数
HIDDEN_SIZE = 128           # Q 网络隐藏层神经元数
REPLAY_SIZE = 10_000        # 经验回放缓冲区最大容量（满了覆盖旧的）
REPLAY_START = 1_000        # 缓冲区至少收集这么多条才开始训练
BATCH_SIZE = 64             # 每次训练从缓冲区随机采样多少条
GAMMA = 0.99                # 折扣因子：未来奖励的衰减率
EPS_START = 1.0             # ε-greedy 初始探索率（100% 随机）
EPS_END = 0.02              # ε 下限（2% 随机）
EPS_DECAY = 2_000           # ε 从初始衰减到下限需要多少步
TARGET_UPDATE = 100         # 每多少步将策略网络参数复制给目标网络
LEARNING_RATE = 0.001       # Adam 学习率

In [ ]:
class DQN(nn.Module):
    """Q 网络：输入状态 → 输出每个动作的 Q 值
    不同于 Ch4 的策略网络（输出概率），DQN 输出的是每个动作的"价值估计"。
    动作选择：直接选 Q 值最大的动作（而非按概率采样）。"""
    def __init__(self, obs_size, hidden_size, n_actions):
        super(DQN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size, hidden_size),  # 输入层：4 维状态（位置、速度、角度、角速度）→ 128 维
            nn.ReLU(),                          # 非线性激活
            nn.Linear(hidden_size, n_actions)  # 输出层：128 维 → 2 维（左推、右推的 Q 值）
        )

    def forward(self, x):
        return self.net(x)  # 输出 [Q(s, 左推), Q(s, 右推)]，不经过 softmax，直接是原始 Q 值</cell>


In [ ]:
# 经验回放缓冲区
# 核心作用：
#   1. 打破样本间的时间相关性（连续帧高度相关，直接训练容易过拟合）
#   2. 提高数据利用率（每条经验可以被多次采样学习）
# 存储 (状态, 动作, 奖励, 下一状态, 是否结束) 的 Transition
Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))

class ReplayBuffer:
    """固定大小的循环缓冲区，满了就覆盖最旧的数据"""
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)  # deque 自动处理容量限制

    def push(self, state, action, reward, next_state, done):
        """存入一条经验"""
        self.buffer.append(Transition(state, action, reward, next_state, done))

    def sample(self, batch_size):
        """随机采样一批经验，返回解包后的张量"""
        transitions = random.sample(self.buffer, batch_size)
        # 转置技巧：把 [(s,a,r,s',d), ...] 变成 ( [s,...], [a,...], ... )
        batch = Transition(*zip(*transitions))
        # 转换为 PyTorch 张量
        states      = torch.FloatTensor(np.array(batch.state))             # (B, 4)
        actions     = torch.LongTensor(batch.action).unsqueeze(1)          # (B, 1) 整数索引
        rewards     = torch.FloatTensor(batch.reward).unsqueeze(1)         # (B, 1)
        next_states = torch.FloatTensor(np.array(batch.next_state))        # (B, 4)
        dones       = torch.FloatTensor(batch.done).unsqueeze(1)           # (B, 1) 1=结束, 0=未结束
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)</cell>


In [ ]:
class Agent:
    """DQN Agent：ε-greedy 探索 + 经验回放 + TD 学习 + 目标网络
    
    DQN 的两大核心技巧（解决 Q-learning 直接套神经网络的发散问题）：
    1. 经验回放（Experience Replay）：随机采样历史经验，打破相关性
    2. 目标网络（Target Network）：冻结一段时间的目标 Q 值计算，稳定 TD 目标
    """
    def __init__(self, env, buffer):
        self.env = env
        self.buffer = buffer
        self.state, _ = self.env.reset()

        # 两个网络：策略网络（持续训练） + 目标网络（定期同步，用于计算稳定的 TD 目标）
        obs_size = env.observation_space.shape[0]  # CartPole: 4
        n_actions = env.action_space.n              # CartPole: 2
        self.policy_net = DQN(obs_size, HIDDEN_SIZE, n_actions)
        self.target_net = DQN(obs_size, HIDDEN_SIZE, n_actions)
        self.target_net.load_state_dict(self.policy_net.state_dict())  # 初始参数相同
        self.target_net.eval()  # 目标网络不训练，只做推理 — 关闭 dropout/batchnorm

        self.optimizer = torch.optim.Adam(self.policy_net.parameters(), lr=LEARNING_RATE)
        self.steps = 0  # 总步数计数器（ε 衰减 + 目标网络同步 都依赖它）

    def epsilon(self):
        """线性衰减 ε：前期多探索，后期多利用
        ε 从 EPS_START(1.0) 线性降到 EPS_END(0.02)，共 EPS_DECAY(2000) 步
        """
        return max(EPS_END, EPS_START - (EPS_START - EPS_END) * self.steps / EPS_DECAY)

    def select_action(self, state, evaluate=False):
        """ε-greedy 动作选择
        evaluate=True  → ε=0，纯贪心（测试/评估时关闭探索）
        evaluate=False → ε 概率随机探索，1-ε 概率选最大 Q 值
        """
        if not evaluate and random.random() < self.epsilon():
            return self.env.action_space.sample()  # 探索：随机选一个动作
        # 利用：选 Q 值最大的动作
        with torch.no_grad():                       # 推理不需要梯度
            state_t = torch.FloatTensor([state])     # 包装成 batch=1 的张量
            q_values = self.policy_net(state_t)      # 用策略网络（非目标网络）估计 Q 值
            return q_values.argmax(dim=1).item()     # 取最大 Q 值的索引

    def learn(self):
        """DQN 核心学习算法（一步 TD 更新）
        1. 从回放缓冲区随机采样 batch
        2. 用目标网络计算 TD 目标：r + γ * max_a' Q_target(s', a') * (1 - done)
        3. 用策略网络计算当前 Q 值
        4. MSE 损失 → 反向传播更新策略网络参数
        """
        if len(self.buffer) < REPLAY_START:
            return  # 缓冲区不够大，先收集足够经验再开始训练

        states, actions, rewards, next_states, dones = self.buffer.sample(BATCH_SIZE)

        # === 计算 TD 目标（Bellman 最优方程的右侧）===
        # TD 目标 = r + γ * max_a' Q_target(s', a')    （非终止状态时）
        # TD 目标 = r                                    （终止状态时，没有未来奖励）
        with torch.no_grad():  # 目标网络不参与梯度计算，保持稳定
            # 用目标网络（非策略网络！）估计下一状态所有动作的 Q 值，取最大值
            next_q = self.target_net(next_states).max(dim=1, keepdim=True).values  # (B, 1)
            # Bellman 方程：当前奖励 + 折扣后的未来最大 Q 值
            target = rewards + GAMMA * next_q * (1 - dones)  # done=1 时清零未来部分

        # === 当前 Q 值：策略网络对当前状态-动作对的估计 ===
        # gather: 从策略网络输出的 Q(s,·) 中取出实际执行的动作 a 对应的 Q(s, a)
        current_q = self.policy_net(states).gather(1, actions)  # (B, 1)

        # === 损失函数：MSE(预测Q值, TD目标) ===
        # 注意：target 是常数（no_grad），梯度只流过 current_q → 策略网络
        loss = nn.MSELoss()(current_q, target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()

    def sync_target(self):
        """硬更新：将策略网络的参数完整复制到目标网络
        频率由 TARGET_UPDATE（每 100 步）控制
        """
        self.target_net.load_state_dict(self.policy_net.state_dict())</cell>


In [5]:
# 训练主循环
import gymnasium as gym

env = gym.make("CartPole-v1")
buffer = ReplayBuffer(REPLAY_SIZE)
agent = Agent(env, buffer)

episode_rewards = []     # 记录每回合奖励
losses = []              # 记录每次训练的损失
best_reward = 0
episode = 0

while True:
    episode += 1
    state, _ = env.reset()
    total_reward = 0

    while True:
        # ① 选动作（ε-greedy）
        action = agent.select_action(state)
        # ② 执行动作
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        # ③ 存入经验
        buffer.push(state, action, reward, next_state, done)
        # ④ 训练一步
        loss = agent.learn()
        if loss is not None:
            losses.append(loss)
        # ⑤ 更新步数 + 同步目标网络
        agent.steps += 1
        if agent.steps % TARGET_UPDATE == 0:
            agent.sync_target()

        total_reward += reward
        state = next_state
        if done:
            break

    episode_rewards.append(total_reward)

    # 每 50 回合打印一次 + 跑评估
    if episode % 50 == 0:
        # 评估：关掉探索，纯贪心跑 20 回合
        eval_rewards = []
        eval_env = gym.make("CartPole-v1")
        for _ in range(20):
            s, _ = eval_env.reset()
            ep_r = 0
            while True:
                a = agent.select_action(s, evaluate=True)
                s, r, term, trunc, _ = eval_env.step(a)
                ep_r += r
                if term or trunc:
                    break
            eval_rewards.append(ep_r)
        eval_env.close()
        avg_eval = np.mean(eval_rewards)
        avg_train = np.mean(episode_rewards[-50:])
        print(f"Ep {episode:4d}: train_reward={avg_train:.1f}, eval_reward={avg_eval:.1f}, "
              f"epsilon={agent.epsilon():.3f}, loss={losses[-1] if losses else 0:.4f}")

        if avg_eval >= 195:
            print(f"\nSolved in {episode} episodes!")
            break

env.close()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/var/folders/fn/fb8vqk716bd4xpgc55y3fqdr0000gn/T/ipykernel_17157/3551401621.py:30: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:281.)
  state_t = torch.FloatTensor([state])


Ep   50: train_reward=18.7, eval_reward=9.1, epsilon=0.543, loss=0.0000
Ep  100: train_reward=14.6, eval_reward=99.1, epsilon=0.184, loss=0.6032
Ep  150: train_reward=126.7, eval_reward=324.3, epsilon=0.020, loss=20.2611

Solved in 150 episodes!
